# Day 21 Tutorial：MLP 与传统模型的同协议比较

> **课程附带教程，不是模型排行榜。** 公开 ESOL 的 logS 结果不代表下游任务性能。

## Goal

读取 Day 07 保存的传统模型配置，让 Dummy、Ridge、受限决策树、随机森林和冻结 MLP 使用相同外部 train/valid、指标和测试封存规则，生成可审计结果表。


## Setup

传统模型参数来自 `curriculum/core/day07_integrated_baseline/reference_baseline/results/run_config.json`，不存在时立即报错而不是静默改用默认值。MLP 使用 Day 20 冻结配方。`fit_seconds` 只计 `fit()`，预测在停止计时后进行。


In [1]:
from pathlib import Path
import contextlib
import io

from rdkit import RDLogger
RDLogger.DisableLog("rdApp.warning")

with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    import deepchem as dc

import numpy as np
import pandas as pd
from IPython.display import display

SEED = 42
np.random.seed(SEED)

def find_repo_root(start=Path.cwd().resolve()):
    for candidate in (start, *start.parents):
        if (candidate / "data" / "public" / "esol.md").exists():
            return candidate
    raise RuntimeError("请从 ML-Learning 仓库内运行本教程。")

REPO_ROOT = find_repo_root()
CACHE_DIR = REPO_ROOT / ".cache" / "deepchem"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

featurizer = dc.feat.CircularFingerprint(size=1024, radius=2)
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    tasks, datasets, transformers = dc.molnet.load_delaney(
        featurizer=featurizer,
        splitter="scaffold",
        transformers=[],
        reload=True,
        data_dir=str(CACHE_DIR),
        save_dir=str(CACHE_DIR),
    )

train_dataset, valid_dataset, _sealed_test_dataset = datasets
X_train = np.asarray(train_dataset.X)
y_train = np.asarray(train_dataset.y).reshape(-1)
X_valid = np.asarray(valid_dataset.X)
y_valid = np.asarray(valid_dataset.y).reshape(-1)
train_ids = np.asarray(train_dataset.ids).astype(str)
valid_ids = np.asarray(valid_dataset.ids).astype(str)

assert transformers == []
assert X_train.shape == (902, 1024) and y_train.shape == (902,)
assert X_valid.shape == (113, 1024) and y_valid.shape == (113,)
print("Task:", tasks[0])
print("Train / valid:", X_train.shape, X_valid.shape)
print("测试集对象保持封存，本教程不创建测试预测。")


Task: measured log solubility in mols per litre
Train / valid: (902, 1024) (113, 1024)
测试集对象保持封存，本教程不创建测试预测。


In [2]:
import json
from time import perf_counter

from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeRegressor

config_path = (
    REPO_ROOT
    / "curriculum"
    / "core"
    / "day07_integrated_baseline"
    / "reference_baseline"
    / "results"
    / "run_config.json"
)
if not config_path.exists():
    raise FileNotFoundError(
        "缺少 Day 07 冻结配置，不能声称传统模型配方一致："
        f"{config_path}"
    )
frozen = json.loads(config_path.read_text(encoding="utf-8"))
frozen_params = frozen["model_params"]

models = {
    "dummy_mean": DummyRegressor(
        **frozen_params["dummy_mean"]
    ),
    "ridge": Ridge(**frozen_params["ridge"]),
    "decision_tree_regularized": DecisionTreeRegressor(
        **frozen_params["decision_tree_regularized"]
    ),
    "random_forest": RandomForestRegressor(
        **frozen_params["random_forest"]
    ),
    "mlp": make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        MLPRegressor(
            hidden_layer_sizes=(32,),
            alpha=0.001,
            early_stopping=True,
            max_iter=300,
            n_iter_no_change=10,
            random_state=SEED,
        ),
    ),
}
assert np.isfinite(X_train).all() and np.isfinite(X_valid).all()
print("Frozen traditional config:", config_path.relative_to(REPO_ROOT))
print("Pre-registered models:", list(models))


Frozen traditional config: curriculum/core/day07_integrated_baseline/reference_baseline/results/run_config.json
Pre-registered models: ['dummy_mean', 'ridge', 'decision_tree_regularized', 'random_forest', 'mlp']


## Steps

### 1. 每个模型拟合一次并统一评价

计时器在 `fit()` 返回后立刻停止。训练指标用于诊断，验证指标用于当前比较；不为任何模型改换样本、公式或搜索预算。


In [3]:
rows = []
for name, model in models.items():
    started = perf_counter()
    model.fit(X_train, y_train)
    fit_seconds = perf_counter() - started

    for split, X_part, y_part in [
        ("train", X_train, y_train),
        ("valid", X_valid, y_valid),
    ]:
        prediction = model.predict(X_part)
        rows.append({
            "model": name,
            "split": split,
            "rmse": root_mean_squared_error(y_part, prediction),
            "mae": mean_absolute_error(y_part, prediction),
            "r2": r2_score(y_part, prediction),
            "fit_seconds": fit_seconds,
        })

results = pd.DataFrame(rows)
valid_rank = (
    results.query("split == 'valid'")
    .sort_values(["rmse", "mae"])
    .reset_index(drop=True)
)
display(valid_rank.round(4))


,model,split,rmse,mae,r2,fit_seconds
0,random_forest,valid,1.7031,1.3124,0.2578,0.1443
1,decision_tree_regularized,valid,2.1106,1.6174,-0.1399,0.0091
2,dummy_mean,valid,2.1714,1.6358,-0.2065,0.0001
3,ridge,valid,2.3795,1.9244,-0.4489,0.0184
4,mlp,valid,3.0735,2.5418,-1.4173,0.3352


### 2. 读取当前差值，而不是宣布普遍赢家

当前第一行只是在一次固定验证协议中的最低 RMSE。单独报告 MLP 与最强传统模型、Dummy 的差值。


In [4]:
traditional_names = {
    "ridge", "decision_tree_regularized", "random_forest"
}
best_traditional = (
    valid_rank[valid_rank["model"].isin(traditional_names)]
    .iloc[0]
)
mlp_row = valid_rank.loc[valid_rank["model"] == "mlp"].iloc[0]
dummy_row = valid_rank.loc[
    valid_rank["model"] == "dummy_mean"
].iloc[0]
print(
    "Current best traditional:",
    best_traditional["model"],
    round(best_traditional["rmse"], 4),
)
print(
    "MLP minus best traditional RMSE:",
    round(mlp_row["rmse"] - best_traditional["rmse"], 4),
)
print(
    "MLP minus Dummy RMSE:",
    round(mlp_row["rmse"] - dummy_row["rmse"], 4),
)
print("These are not statistical-significance claims.")


Current best traditional: random_forest 1.7031
MLP minus best traditional RMSE: 1.3704
MLP minus Dummy RMSE: 0.9022
These are not statistical-significance claims.


## Checks

每个候选必须恰好有 train/valid 两行，指标有限，测试集没有进入结果表；传统模型对象参数必须匹配保存配置。


In [5]:
assert results.groupby(["model", "split"]).size().eq(1).all()
assert set(results["model"]) == set(models)
assert set(results["split"]) == {"train", "valid"}
assert np.isfinite(
    results[["rmse", "mae", "r2", "fit_seconds"]]
).all().all()
assert "test" not in set(results["split"])
assert models["ridge"].get_params() == frozen_params["ridge"]
assert (
    models["decision_tree_regularized"].get_params()
    == frozen_params["decision_tree_regularized"]
)
assert (
    models["random_forest"].get_params()
    == frozen_params["random_forest"]
)
print("Frozen-config and fair-comparison checks passed.")


Frozen-config and fair-comparison checks passed.


## Next Steps

保存你亲自运行的完整候选与环境，再用多种子/交叉验证检查排名。Day 22 用分组玩具数据拆解训练内预测泄漏；Day 23 以相同随机森林和 MLP 配方实现 scaffold-aware OOF stacking。
